# 02 — Feature Engineering

**NovaFin Group Capstone · ePGD MLDS, IIIT Bombay · Group 2 · Dipesh Kumar Yadav**

Four things happen here, in order:

1. build features for all eight modules,
2. **prove** — not assert — that none of them can see the future,
3. demonstrate why target encoding must be fitted out of fold,
4. run the three-stage selection filter and persist the artefacts.

> Prerequisites: `00_environment_check.ipynb`, then `01_setup_and_eda.ipynb`.

**The rule every feature obeys:** a feature for row *t* may use rows *< t* only.
In pandas that is one habit — **shift before you roll**. The single forward
reference in the entire codebase is `causal.forward_target`, which builds
*targets*. You can `grep` for it.

## 1 · Preamble

In [ ]:
import os

os.environ["PYTHONHASHSEED"] = "42"

IN_COLAB = "google.colab" in str(get_ipython())  # noqa: F821
if IN_COLAB:
    import subprocess, sys
    from pathlib import Path

    REPO_URL = "https://github.com/yadavdipesh/novafin-capstone.git"
    if not Path("/content/novafin-capstone").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/novafin-capstone"], check=True)
    os.chdir("/content/novafin-capstone")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["NOVAFIN_DATA_RAW"] = (
        "/content/drive/MyDrive/ePGD - MLDS IIT Bombay/C5 ML In Finanace/Data"
    )

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from novafin import viz
from novafin.config import load_config
from novafin.data import load_all, load_dataset, make_feature_frame
from novafin.features import (
    CUSTOMER_DECISION_COLUMNS, CyclicalEncoder, FrequencyEncoder,
    OutOfFoldTargetEncoder, PRIMITIVE_FEATURES, assert_causal, build_features,
    build_preprocessor, correlation_filter, select_features, variance_filter,
)
from novafin.features.causal import CausalityViolation
from novafin.utils.io import dump_pickle
from novafin.utils.logging_utils import setup_logging
from novafin.utils.seed import seed_everything
from novafin.utils.theme import apply_theme, save_figure

cfg = load_config()
setup_logging("WARNING", log_file=cfg.paths.logs / "02_features.log")
seed_everything(cfg.reproducibility.seed)
apply_theme()

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
print("config fingerprint:", cfg.fingerprint())

## 2 · Build features for every module

One builder per module, dispatched from a registry. Each returns a
`FeatureResult` carrying the frame, the engineered column list, the target name
and any rows dropped (a forward target costs you the last observation of every
entity — that is arithmetic, not a bug).

In [ ]:
results = load_all(cfg=cfg)
features = {}
rows = []

for key, result in results.items():
    built = build_features(key, result.frame, cfg)
    features[key] = built
    rows.append(built.summary())

overview = pd.DataFrame(rows)
overview

In [ ]:
for key, built in features.items():
    print(f"\n{key.upper()}  target = {built.target}")
    for note in built.notes:
        print(f"    - {note}")

### 2.1 · The leakage boundary

`make_feature_frame` is the only sanctioned way to build **X**. It removes every
column in `forbidden_features` — identifiers, the entity key, the target,
secondary/derived targets, and the audited leaks — so a model cannot see them
even if a notebook forgets.

In [ ]:
matrices = {}
shape_rows = []

for key, built in features.items():
    spec = cfg.dataset(key)
    extra = [c for c in ("fwd_return_5d", "fwd_inflows_5d") if c in built.frame.columns]
    X, _ = make_feature_frame(built.frame, spec, extra_drop=extra)
    # Datetime columns have been consumed by the builders; drop the raw stamps.
    X = X.drop(columns=[c for c in X.columns
                        if pd.api.types.is_datetime64_any_dtype(X[c])], errors="ignore")
    y = built.frame[built.target]
    matrices[key] = (X, y)
    shape_rows.append({
        "module": key, "raw_cols": results[key].frame.shape[1],
        "after_engineering": built.frame.shape[1], "in_X": X.shape[1],
        "removed_by_boundary": built.frame.shape[1] - X.shape[1],
        "target": built.target,
    })

pd.DataFrame(shape_rows)

## 3 · The causality proof

This is the strongest validation artefact in the repository, and the one to put
on a slide.

**Method.** Build features. Corrupt *only* the final 20% of rows — sign-flip,
multiply by 1000, add noise. Rebuild. Assert every model feature in the
untouched prefix is **bit-identical**.

If any feature peeked forward, its earlier values would move. Nothing passes
this by accident.

Panel datasets are sliced to a single entity first, so that row order equals
time order — otherwise "the last 20% of rows" would mean "three whole tickers",
which is a different test.

In [ ]:
PERTURB = {
    "initiatives": ["Revenue_Year1", "Revenue_Year3", "Initial_Investment", "Expected_ROI"],
    "loans": ["Loan_Amount", "Collateral_Value", "Annual_Income", "Interest_Rate"],
    "transactions": ["Amount", "Historical_Avg_Transaction", "Fraud_Flag", "Timestamp"],
    "customers": ["Account_Balance", "Annual_Income", "Complaints", "Estimated_CLV"],
    "market": ["Return", "Close", "Volume", "VIX", "Momentum_20D", "Market_Return"],
    "liquidity": ["Expected_Outflows", "Expected_Inflows", "Deposits", "Market_Stress"],
    "options": ["Spot", "Strike", "Volatility", "Market_Price", "Black_Scholes_Price"],
    "hft": ["Mid_Price", "OBI_Level1", "OBI_3Level", "Trade_Volume", "Relative_Spread"],
}
SINGLE_ENTITY = {"market": ("Ticker", "RELIANCE"), "hft": ("Stock", "NOVA01")}

proof_rows = []
for key in PERTURB:
    frame = results[key].frame
    if key in SINGLE_ENTITY:
        column, value = SINGLE_ENTITY[key]
        frame = frame[frame[column] == value].reset_index(drop=True)

    spec = cfg.dataset(key)
    built = build_features(key, frame, cfg)
    ignore = set(spec.forbidden_features) | {built.target, "fwd_return_5d", "fwd_inflows_5d"}
    checked = [c for c in built.engineered if c not in ignore]

    try:
        assert_causal(
            lambda data, k=key: build_features(k, data, cfg).frame,
            frame, perturb_columns=PERTURB[key],
            ignore_columns=ignore, tail_fraction=0.20,
        )
        verdict, detail = "PASS", ""
    except CausalityViolation as exc:
        verdict, detail = "FAIL", str(exc).splitlines()[0]

    proof_rows.append({
        "module": key, "rows_tested": len(frame),
        "features_checked": len(checked), "verdict": verdict, "detail": detail,
    })

proof = pd.DataFrame(proof_rows)
display(proof)
print(f"\nTotal model features proved causal: {proof.loc[proof.verdict == 'PASS', 'features_checked'].sum()}")
assert (proof["verdict"] == "PASS").all(), "a feature builder looks ahead"

### 3.1 · Show that the test can actually fail

A test that never fails proves nothing. Here is a deliberately leaky builder —
the proof rejects it.

In [ ]:
demo = pd.DataFrame({"x": np.arange(200, dtype="float64")})

def leaky_builder(data):
    out = data.copy()
    out["peeks_forward"] = out["x"].shift(-1)          # looks one row ahead
    out["leaky_rolling"] = out["x"].rolling(5, min_periods=1).mean()   # includes row t
    return out

def causal_builder(data):
    out = data.copy()
    out["lag_1"] = out["x"].shift(1)
    out["roll_5"] = out["x"].shift(1).rolling(5, min_periods=1).mean()  # shift THEN roll
    return out

try:
    assert_causal(leaky_builder, demo, perturb_columns=["x"])
    print("UNEXPECTED: the leaky builder passed")
except CausalityViolation as exc:
    print("Leaky builder correctly REJECTED:")
    print("   ", str(exc).splitlines()[0])

report = assert_causal(causal_builder, demo, perturb_columns=["x"])
print("\nCausal builder PASSED (empty offender report):", report.empty)

### 3.2 · The one-line difference that causes the leak

`rolling` then `shift` versus `shift` then `rolling`. The values differ, and the
wrong one silently includes the current row in its own average.

In [ ]:
toy = pd.DataFrame({"entity": ["A"] * 10, "value": list(range(10))})

leaky = toy.groupby("entity")["value"].transform(lambda s: s.rolling(3, min_periods=1).mean())
correct = toy.groupby("entity")["value"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())

comparison = pd.DataFrame({
    "value": toy["value"],
    "rolling(3).mean()  [LEAKY]": leaky,
    "shift(1).rolling(3).mean()  [CORRECT]": correct,
})
display(comparison)
print("Row 5: value = 5.  Leaky mean = mean(3,4,5) = 4.0  <- sees itself")
print("                   Correct mean = mean(2,3,4) = 3.0")

## 4 · Encoding without leaking

Every statistic used to transform a feature — an imputation mean, a scaler's
standard deviation, a one-hot category list, a category-level target mean — is a
**fitted parameter**. Fitting it on the full dataset lets test rows shape the
training representation.

### 4.1 · Why target encoding needs out-of-fold fitting

Naive target encoding replaces a category with the mean target of its rows —
letting every row see its own label. On a high-cardinality column the model
memorises the training set and the CV score looks superb.

The demonstration below uses `Device_ID` (1,500 levels) against `Fraud_Flag`.

In [ ]:
txn = results["transactions"].frame.copy()
txn["Device_Cat"] = txn["Device_ID"].astype(str)
y_fraud = txn["Fraud_Flag"].astype("float64")

naive = y_fraud.groupby(txn["Device_Cat"]).transform("mean")

encoder = OutOfFoldTargetEncoder(columns=["Device_Cat"], n_splits=5, smoothing=10.0)
oof = encoder.fit_transform(txn[["Device_Cat"]], y_fraud)["Device_Cat"]

print(f"levels in Device_ID            : {txn['Device_Cat'].nunique():,}")
print(f"prior (overall fraud rate)     : {encoder.prior_:.5f}")
print()
print(f"NAIVE encoding  |corr| with target = {abs(np.corrcoef(naive, y_fraud)[0, 1]):.4f}   <-- LEAK")
print(f"OUT-OF-FOLD     |corr| with target = {abs(np.corrcoef(oof, y_fraud)[0, 1]):.4f}   <-- honest")
print()
print("The naive figure is manufactured by each row seeing its own label.")
print("A model trained on it would score brilliantly in CV and fail in production.")

fig = viz.plot_leakage_evidence(
    np.array([abs(np.corrcoef(naive, y_fraud)[0, 1])]),
    np.array([abs(np.corrcoef(oof, y_fraud)[0, 1])]),
    labels=("Naive target encoding", "Out-of-fold"),
    title="Target encoding: apparent signal from a 1,500-level ID",
    ylabel="|correlation| with Fraud_Flag",
)
save_figure(fig, "02_target_encoding_leak", close=False)
plt.show()

### 4.2 · Smoothing, unseen categories, cyclical time

In [ ]:
# Smoothing pulls small categories toward the prior, so one observation
# cannot dictate an encoding.
tiny = pd.DataFrame({"cat": ["rare"] + ["common"] * 999})
tiny_y = pd.Series([1.0] + [0.0] * 999)
smoothed = OutOfFoldTargetEncoder(columns=["cat"], smoothing=10.0).fit(tiny, tiny_y)
print("category 'rare' has ONE row with target 1.0")
print(f"   unsmoothed mean would be 1.000")
print(f"   smoothed encoding is      {smoothed.transform(tiny)['cat'].iloc[0]:.4f}  (prior = {smoothed.prior_:.4f})")

# Unseen categories fall back to the prior rather than raising.
unseen = smoothed.transform(pd.DataFrame({"cat": ["never_seen"]}))["cat"].iloc[0]
print(f"\nunseen category -> {unseen:.4f} (the prior)")

# Frequency encoding: rarity as signal, no target involvement at all.
freq = FrequencyEncoder(columns=["Merchant_Category"]).fit(txn[["Merchant_Category"]])
print("\nmerchant category frequencies (fitted on training rows only):")
print(freq.frequencies_["Merchant_Category"].round(4).to_string())

# Cyclical encoding: hour 23 is adjacent to hour 0.
hours = pd.DataFrame({"hour": [0, 6, 12, 18, 23]})
print("\ncyclical hour encoding:")
display(CyclicalEncoder({"hour": 24}).fit_transform(hours).round(3))

### 4.3 · The preprocessor

`build_preprocessor` returns an **unfitted** `ColumnTransformer`. That is
deliberate: returning a transformed matrix would make the leaky usage the easy
one. It is fitted inside each CV fold in Phase 4.

In [ ]:
X_loans, y_loans = matrices["loans"]
preprocessor = build_preprocessor(X_loans, scale=True)

print(type(preprocessor).__name__)
for name, _, columns in preprocessor.transformers:
    print(f"  {name:<12} {len(columns):>3} column(s)")

# Fitted on a TRAIN slice only, purely to show the output width.
cut = int(len(X_loans) * 0.8)
transformed = preprocessor.fit(X_loans.iloc[:cut]).transform(X_loans.iloc[cut:])
print(f"\nfitted on {cut:,} training rows -> {transformed.shape[1]} output columns")
print("(one-hot expansion of credit_band and Customer_Type accounts for the increase)")

## 5 · Feature selection

Three filters, cheapest first:

| Stage | Cost | Removes |
|---|---|---|
| variance | O(n) | constants and 99.5%-dominant flags |
| correlation | O(k²) | one of each redundant pair |
| null importance | ~26 model fits | features no better than a shuffled target |

Null importance is the only non-heuristic stage. It compares each feature's real
importance against **its own** distribution under a shuffled target — so a
high-cardinality column that scores well merely because it offers many split
points is correctly rejected. `Sector_Risk` (3,978 distinct values, correlation
0.029 with default) is the live test case.

In [ ]:
cheap_rows = []
for key, (X, y) in matrices.items():
    kept_v, dropped_v = variance_filter(X)
    kept_c, dropped_c = correlation_filter(X[kept_v], threshold=0.95)
    cheap_rows.append({
        "module": key, "start": X.shape[1],
        "after_variance": len(kept_v), "after_correlation": len(kept_c),
        "dropped_variance": len(dropped_v), "dropped_correlation": len(dropped_c),
    })
pd.DataFrame(cheap_rows)

In [ ]:
# What exactly was removed, for the credit module.
X_loans, y_loans = matrices["loans"]
kept_v, dropped_v = variance_filter(X_loans)
kept_c, dropped_c = correlation_filter(X_loans[kept_v], threshold=0.95)

print("VARIANCE stage:")
for column, reason in dropped_v.items():
    print(f"   drop {column:<28} {reason}")
print("\nCORRELATION stage:")
for column, reason in dropped_c.items():
    print(f"   drop {column:<28} {reason}")

### 5.1 · Null importance — does the pipeline find the noise column?

`Sector_Risk` is uniform noise with essentially zero relationship to default.
A pipeline that identifies and removes it is a far better exhibit than one that
merely tolerates it.

In [ ]:
report = select_features(
    X_loans, y_loans, task="classification",
    correlation_threshold=0.95, run_null_importance=True, n_runs=25,
    random_state=cfg.reproducibility.seed,
)
print(report.summary())

scores = report.scores
display(scores.head(15).round(2))

print("\nSector_Risk verdict:")
row = scores[scores["feature"] == "Sector_Risk"]
if len(row):
    display(row.round(3))
print("  removed by null importance:", "Sector_Risk" in report.dropped)
if "Sector_Risk" in report.dropped:
    print("  reason:", report.dropped["Sector_Risk"])

In [ ]:
# Visualise actual importance against the null distribution.
top = scores.head(20).sort_values("gain_over_null")
fig, ax = plt.subplots(figsize=(9, 6.5))
ax.barh(top["feature"], top["actual_importance"], color="#00B2A9", label="actual")
ax.barh(top["feature"], top["null_p95"], color="#B88F20", alpha=0.75,
        height=0.45, label="95th percentile of shuffled-target null")
ax.legend()
ax.set_title("Null importance — M2 credit risk")
ax.set_xlabel("LightGBM importance")
save_figure(fig, "02_null_importance_credit", close=False)
plt.show()

## 6 · Module-specific checks

### 6.1 · M10 — our Black-Scholes against the supplied column

Judged on **relative** error. The CSV stores `Volatility`, `Interest_Rate` and
`Time_to_Maturity` rounded to 4 decimal places, and a two-year option has rho in
the thousands — so a 5×10⁻⁵ input rounding moves the price by ~0.2 in absolute
terms while the implementation remains correct to ~0.01%.

In [ ]:
opt = features["options"].frame
priced = opt[opt["Black_Scholes_Price"] > 1.0]
relative = priced["bs_check_error_pct"].abs()

print(f"relative error vs supplied Black_Scholes_Price (n = {len(priced):,}):")
print(f"   median {relative.median():.6f}   p90 {relative.quantile(0.90):.6f}   "
      f"p99 {relative.quantile(0.99):.6f}   max {relative.max():.6f}")
print("\nOur implementation agrees with the dataset's own pricing to ~0.01% at the median.")
print("That cross-validates BOTH - and gives us the Greeks the file does not ship.")

print("\nMispricing target (Market_Price - Black_Scholes_Price):")
print(opt["mispricing"].describe(percentiles=[.01, .5, .99]).round(4).to_string())
print("\nFair-fight feature set (exactly what Black-Scholes itself sees):")
print("  ", PRIMITIVE_FEATURES)

### 6.2 · M4 — decision columns are not model features

The population percentile ranks answer *"which 1,000 customers do we contact?"*
over the whole book at decision time. They are a **business ranking**, and a
percentile computed across every row leaks across the train/test split — so they
are listed in `drop_always` and removed from X automatically.

The causality proof caught this; the fix was to rename them `decision_*` and
exclude them, not to delete the business logic.

In [ ]:
X_cust, _ = matrices["customers"]
print("decision columns:", CUSTOMER_DECISION_COLUMNS)
print("present in X    :", [c for c in CUSTOMER_DECISION_COLUMNS if c in X_cust.columns])
assert not [c for c in CUSTOMER_DECISION_COLUMNS if c in X_cust.columns]

cust = features["customers"].frame
top_1000 = cust.nlargest(cfg.fin("churn", "contact_budget", default=1000),
                         "decision_priority_score")
print(f"\nTop {len(top_1000):,} customers by decision_priority_score:")
print(f"   mean Estimated_CLV : {top_1000['Estimated_CLV'].mean():,.0f} "
      f"(book average {cust['Estimated_CLV'].mean():,.0f})")
print(f"   mean complaints    : {top_1000['Complaints'].mean():.2f} "
      f"(book average {cust['Complaints'].mean():.2f})")
print(f"   CLV captured       : {top_1000['Estimated_CLV'].sum() / cust['Estimated_CLV'].sum():.1%} "
      f"of total book value from {len(top_1000) / len(cust):.0%} of customers")

### 6.3 · M3 — the strongest engineered fraud features

In [ ]:
from novafin.data import decile_lift

txn_features = features["transactions"].frame
for feature in ["amount_vs_history", "amount_z_vs_cust", "amount_vs_device_mean"]:
    subset = txn_features[[feature, "Fraud_Flag"]].dropna()
    lift = decile_lift(subset, feature, "Fraud_Flag")
    print(f"\n{feature}: fraud rate {lift['event_rate'].iloc[0]:.4f} (D1) -> "
          f"{lift['event_rate'].iloc[-1]:.4f} (D10), lift {lift['lift'].iloc[-1]:.2f}x")
    fig = viz.plot_decile_lift(lift, feature, "Fraud_Flag")
    save_figure(fig, f"02_m03_{feature}_lift", close=False)
    plt.show()

## 7 · Persist

Feature frames go to `data/processed/` as parquet so Phase 4 does not rebuild
them; selection reports and shape tables go to `reports/tables/`.

In [ ]:
cfg.paths.data_processed.mkdir(parents=True, exist_ok=True)
cfg.paths.tables.mkdir(parents=True, exist_ok=True)

def persist(frame, stem):
    """Parquet when an engine is available, CSV otherwise.

    Parquet preserves dtypes and is ~4x smaller, which matters for the
    120k-row order book. pyarrow is pinned in requirements.txt, so the
    fallback should never fire in Colab - but a persistence step must not
    be the thing that breaks a run.
    """
    parquet_path = cfg.paths.data_processed / f"{stem}.parquet"
    try:
        frame.to_parquet(parquet_path, index=False)
        return parquet_path
    except ImportError:
        csv_path = cfg.paths.data_processed / f"{stem}.csv"
        frame.to_csv(csv_path, index=False)
        return csv_path

written = [persist(built.frame, f"{key}_features") for key, built in features.items()]

overview.to_csv(cfg.paths.tables / "02_feature_overview.csv", index=False)
pd.DataFrame(shape_rows).to_csv(cfg.paths.tables / "02_feature_shapes.csv", index=False)
proof.to_csv(cfg.paths.tables / "02_causality_proof.csv", index=False)
pd.DataFrame(cheap_rows).to_csv(cfg.paths.tables / "02_selection_stages.csv", index=False)
report.to_frame().to_csv(cfg.paths.tables / "02_selection_credit.csv", index=False)
scores.to_csv(cfg.paths.tables / "02_null_importance_credit.csv", index=False)

print("Feature frames:")
for path in sorted(written):
    print(f"   {path.name:<34} {path.stat().st_size / 1024**2:6.2f} MB")
print("\nTables:")
for path in sorted(cfg.paths.tables.glob("02_*.csv")):
    print("   ", path.name)

---

## Phase 3 summary

| Guarantee | Established by |
|---|---|
| No feature can see the future | `assert_causal` — corrupt the future, prefix bit-identical |
| The proof can fail | leaky builder demonstrated and rejected in §3.1 |
| Targets are constructed, not read | `forward_target` is the only forward reference |
| Encoding cannot leak | `OutOfFoldTargetEncoder`; preprocessor returned **unfitted** |
| Dataset-wide statistics excluded | `decision_*` columns in `drop_always` |
| Noise is removed, not tolerated | null-importance filter vs a shuffled target |

**NEXT:** `03_baseline_models` — Level 1 of the fine-tuning ladder: baselines
per module, cross-validated with the Phase-2 splitters, logged to MLflow, and
the first real numbers for the results table.